In [14]:
!pip install -q pdfplumber pymupdf python-docx google-api-python-client

In [15]:
# =========================
# IMPORTS
# =========================
import os
import io
import json
import tempfile
from datetime import datetime

import pdfplumber
import fitz  # PyMuPDF
from docx import Document

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

In [16]:
# Authenticate Google Drive
auth.authenticate_user()
drive_service = build("drive", "v3")

FOLDER_NAME = "Research"
OUTPUT_FILE_NAME = "research_context.json"

In [17]:
def table_to_markdown(rows):
    if not rows:
        return ""

    cleaned = []
    for row in rows:
        cleaned.append([
            str(cell).replace("\n", " ").strip() if cell else ""
            for cell in row
        ])

    max_cols = max(len(r) for r in cleaned)
    cleaned = [r + [""] * (max_cols - len(r)) for r in cleaned]

    header = cleaned[0]
    body = cleaned[1:]

    md = []
    md.append("| " + " | ".join(header) + " |")
    md.append("| " + " | ".join(["---"] * max_cols) + " |")

    for row in body:
        md.append("| " + " | ".join(row) + " |")

    return "\n".join(md)

In [18]:
def find_folder_id(name):
    res = drive_service.files().list(
        q=f"name='{name}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
        fields="files(id,name)"
    ).execute()

    return res["files"][0]["id"]


def list_files(folder_id):
    res = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id,name,mimeType,webViewLink)"
    ).execute()

    return res.get("files", [])


def download_file(file_id, name):
    request = drive_service.files().get_media(fileId=file_id)
    path = os.path.join(tempfile.gettempdir(), name)

    with io.FileIO(path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

    return path


def export_gdoc(file_id, name):
    request = drive_service.files().export_media(
        fileId=file_id,
        mimeType="application/vnd.openxmlformats-officedocument.wordprocessingml.document"
    )

    path = os.path.join(tempfile.gettempdir(), name + ".docx")

    with io.FileIO(path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

    return path

In [19]:
def extract_pdf(path):
    pages = []

    doc = fitz.open(path)

    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages):
            page_num = i + 1

            text = doc[i].get_text("text").strip()

            tables = []
            for t_idx, table in enumerate(page.extract_tables() or [], start=1):
                rows = [
                    [cell.strip() if isinstance(cell, str) else "" for cell in row]
                    for row in table
                ]

                tables.append({
                    "table_number": t_idx,
                    "rows": rows,
                    "markdown": table_to_markdown(rows)
                })

            pages.append({
                "page_number": page_num,
                "text": text,
                "tables": tables
            })

    return {"type": "pdf", "pages": pages}

In [20]:
def extract_docx(path):
    doc = Document(path)

    blocks = []

    for p in doc.paragraphs:
        txt = p.text.strip()
        if txt:
            blocks.append({
                "type": "paragraph",
                "text": txt
            })

    for t_idx, table in enumerate(doc.tables, start=1):
        rows = [
            [cell.text.strip() for cell in row.cells]
            for row in table.rows
        ]

        blocks.append({
            "type": "table",
            "table_number": t_idx,
            "rows": rows,
            "markdown": table_to_markdown(rows)
        })

    return {"type": "docx", "blocks": blocks}

In [21]:
def upload_json(folder_id, data):
    path = os.path.join(tempfile.gettempdir(), OUTPUT_FILE_NAME)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    media = MediaFileUpload(path, mimetype="application/json")

    drive_service.files().create(
        body={"name": OUTPUT_FILE_NAME, "parents": [folder_id]},
        media_body=media
    ).execute()

In [22]:
folder_id = find_folder_id(FOLDER_NAME)
files = list_files(folder_id)

result = {
    "generated_at": datetime.utcnow().isoformat(),
    "documents": []
}

for file in files:
    name = file["name"]
    mime = file["mimeType"]
    file_id = file["id"]

    if name == OUTPUT_FILE_NAME:
        continue

    print("Processing:", name)

    record = {
        "file_name": name,
        "file_id": file_id,
        "mime_type": mime,
        "drive_url": file.get("webViewLink"),
        "status": "success"
    }

    try:
        if mime == "application/pdf":
            path = download_file(file_id, name)
            record["content"] = extract_pdf(path)

        elif mime == "application/vnd.openxmlformats-officedocument.wordprocessingml.document":
            path = download_file(file_id, name)
            record["content"] = extract_docx(path)

        elif mime == "application/vnd.google-apps.document":
            path = export_gdoc(file_id, name)
            record["content"] = extract_docx(path)

        else:
            record["status"] = "skipped"

    except Exception as e:
        record["status"] = "error"
        record["error"] = str(e)

    result["documents"].append(record)

upload_json(folder_id, result)

print("✅ Done. JSON saved to Drive.")

/tmp/ipykernel_8707/3900859333.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat(),


Processing: 2005.11401v4.pdf
Processing: 2305.10601v2.pdf
Processing: 2302.04761v1.pdf
Processing: 2210.03629v3.pdf
Processing: 1706.03762v7.pdf
✅ Done. JSON saved to Drive.
